# Lekcija 09 - Metakognitivni vzorec oblikovanja


## Setup

This notebook demonstrates the Metacognition design pattern using the Microsoft Agent Framework.

**Prerequisites:**
- Azure OpenAI deployment configured via environment variables
- Azure CLI authenticated (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Kaj je metakognicija?

Metakognicija je **razmišljanje o razmišljanju**. V kontekstu AI agentov to pomeni ustvarjanje agentov, ki lahko:

- **Samo-reflektirajo** svoja lastna izhoda in proces sklepanja
- **Zaznavajo napake** in se lepo opomorejo namesto tihega spodrsljaja
- **Ocenjujejo**, ali so njihovi odgovori popolni in uporabni
- **Prilagajajo** svojo strategijo, ko začetni pristop ne deluje (npr. prehod na rezervni sistem)

Agent z metakognicijo ne odgovarja samo na vprašanja — spremlja svoje lastno delovanje in se na tekoče prilagaja.


## Primarna in rezervna orodja

Pogost vzorec metakognicije je **strategija rezerve**. Agent najprej preizkusi primarno orodje; če to ne uspe (npr. napaka 404), agent prepozna neuspeh in pregledno preklopi na rezervno orodje.

To odraža sisteme iz resničnega sveta, kjer primarne storitve morda niso na voljo in morajo agenti sami diagnosticirati težavo, preden izberejo alternativno pot.

Spodaj definiramo dve orodji za iskanje letov:
- **Primarno** — pokriva Pariz, Tokio in Barcelono
- **Rezervno** — pokriva Berlin, Sydney in New York City


In [ ]:
@tool(approval_mode="never_require")
def get_flight_times(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times for a destination (primary source)."""
    flights = {
        "Paris": "Departures: 08:00, 12:30, 17:45 — from $350",
        "Tokyo": "Departures: 11:00, 23:30 — from $890",
        "Barcelona": "Departures: 07:15, 14:00, 19:30 — from $280",
    }
    if destination in flights:
        return flights[destination]
    raise Exception(f"404: No flights found for {destination} in primary system")


@tool(approval_mode="never_require")
def get_flight_times_backup(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times from backup system (used when primary fails)."""
    backup_flights = {
        "Berlin": "Departures: 09:00, 16:00 — from $220",
        "Sydney": "Departures: 22:00 — from $1200",
        "New York City": "Departures: 06:00, 10:30, 15:00, 20:00 — from $450",
    }
    return backup_flights.get(
        destination,
        f"No flights found for {destination} in any system. Please try again later.",
    )

## Samoreflektirajoči agent z obnovitvijo po napaki

Spodnjemu agentu je naročeno, naj najprej preizkusi primarni letalski sistem, prepozna napake in pregledno preklopi na rezervni sistem. Po vsakem odgovoru se na kratko sam ovrednoti, ali je v celoti odgovoril na uporabnikovo vprašanje.


In [ ]:
agent = client.as_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="FlightBookingAgent",
    instructions="""You are a flight booking agent with self-reflection capabilities.

When looking up flights:
1. Try the primary flight system first (get_flight_times)
2. If the primary system fails (404 error), acknowledge the error and try the backup system (get_flight_times_backup)
3. Always explain to the user what happened — be transparent about fallbacks
4. If both systems fail, apologize and suggest alternatives

After each response, briefly evaluate whether your answer was complete and helpful.""",
)

# Test with a destination in primary system
print("=== Test 1: Destination in primary system ===")
response = await agent.run(
    "What flights are available to Paris?",
    )
print(response)

# Test with a destination only in backup system
print("\n=== Test 2: Destination only in backup system ===")
response = await agent.run(
    "What flights are available to Berlin?",
    )
print(response)

## Vzorec samoevalvacije

Drugi vidik metakognicije je **samoevalvacija**: ločen agent (ali isti agent v drugem pregledu) pregleda odgovor glede na popolnost, točnost in koristnost.

Spodaj ustvarimo agenta `ResponseEvaluator`, ki ocenjuje odgovore potovalnih agentov po treh dimenzijah.


In [ ]:
evaluation_agent = client.as_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="ResponseEvaluator",
    instructions="""You are a quality evaluator for travel agent responses.
Given a travel question and the agent's response, evaluate:
1. Completeness: Did it answer all parts of the question? (1-5)
2. Accuracy: Is the information correct? (1-5)
3. Helpfulness: Would a traveler find this useful? (1-5)
Provide a brief evaluation with scores and one suggestion for improvement.""",
)

# Evaluate the agent's response from Test 1
eval_prompt = f"""Question: What flights are available to Paris?
Agent Response: {response}

Please evaluate the above response."""

evaluation = await evaluation_agent.run(eval_prompt)
print("=== Self-Evaluation ===")
print(evaluation)

## Povzetek

V tem poglavju ste se naučili, kako zgraditi **metakognitivne agente** z uporabo Microsoft Agent Framework:

- **Samorefleksija**: Agenti, ki spremljajo lastno razmišljanje in transparentno sporočajo, kaj se je zgodilo.
- **Obnova po napakah z nadomestnimi rešitvami**: Vzorec primarnega in rezervnega orodja, kjer agent zazna napake (npr. napake 404) in samodejno poskusi alternativen vir.
- **Samoocenjevanje**: Poseben ocenjevalni agent, ki ocenjuje odzive glede na popolnost, natančnost in koristnost.

Ti vzorci naredijo agente bolj robustne, transparentne in zaupanja vredne — ključne lastnosti za produkcijsko uporabo.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Omejitev odgovornosti**:
Ta dokument je bil preveden z uporabo AI prevajalske storitve [Co-op Translator](https://github.com/Azure/co-op-translator). Čeprav si prizadevamo za natančnost, vas prosimo, da upoštevate, da avtomatizirani prevodi lahko vsebujejo napake ali netočnosti. Izvirni dokument v njegovem izvirnem jeziku je treba obravnavati kot avtoritativni vir. Za kritične informacije je priporočljiv strokovni človeški prevod. Ne odgovarjamo za morebitna nesporazume ali napačne interpretacije, ki izhajajo iz uporabe tega prevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
